## 1. Import Libraries and Setup

In [6]:
import sys
import os
import json
import pandas as pd
import time
from pathlib import Path    
from typing import Dict, List
from tqdm import tqdm

# Add project root to path
sys.path.insert(0, str(Path.cwd()))

# Import services from workspace
from app.main_optimized import OptimizedShoppingCartPipeline
from app.services.ontology_service import OntologyService
from app.services.bedrock_kb_service import BedrockKBService
from app.services.ingredient_resolver import IngredientResolver
from dotenv import load_dotenv

# Load environment
load_dotenv()

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 2. Initialize Services and Load Knowledge Base

In [7]:
# Initialize pipeline and services
print("Initializing AI Service Pipeline...")
pipeline = OptimizedShoppingCartPipeline(max_workers=3, recipe_cache_ttl=3600)
ontology = OntologyService()
kb_service = BedrockKBService()
ingredient_resolver = IngredientResolver(ontology)

print(f"Pipeline initialized")
print(f"   - Total ingredients in KB: {len(ontology.ingredients)}")
print(f"   - Total dishes in KB: {len(ontology.dishes)}")

# Load ground truth knowledge bases
kb_path = Path("app/data/knowledge_base")

with open(kb_path / "dish_knowledge_base.json", "r", encoding="utf-8") as f:
    dish_kb = json.load(f)

with open(kb_path / "ingredient_knowledge_base.json", "r", encoding="utf-8") as f:
    ingredient_kb = json.load(f)

print(f"\nLoaded Ground Truth:")
print(f"   - Dishes: {len(dish_kb)}")
print(f"   - Ingredients: {len(ingredient_kb)}")

Initializing AI Service Pipeline...
Pipeline initialized
   - Total ingredients in KB: 8112
   - Total dishes in KB: 10741

Loaded Ground Truth:
   - Dishes: 10741
   - Ingredients: 8112

Loaded Ground Truth:
   - Dishes: 10741
   - Ingredients: 8112


## 3. Configuration

In [8]:
# Cấu hình các tham số

SAMPLE_SIZE = 30

# Core ingredient threshold
MIN_IMPORTANCE_CORE = 2

# Cache settings
CACHE_FILE = 'predictions_cache_v1.json'
USE_CACHE_AUTO = False 

# API delay 
API_DELAY = 0.5 

# Random seed for reproducibility
RANDOM_SEED = 42

print("✅ Configuration loaded:")
print(f"   • Sample size: {SAMPLE_SIZE} dishes")
print(f"   • Core importance threshold: >= {MIN_IMPORTANCE_CORE}")
print(f"   • Cache file: {CACHE_FILE}")
print(f"   • Auto use cache: {USE_CACHE_AUTO}")
print(f"   • API delay: {API_DELAY}s")
print(f"   • Random seed: {RANDOM_SEED}")

✅ Configuration loaded:
   • Sample size: 30 dishes
   • Core importance threshold: >= 2
   • Cache file: predictions_cache_v1.json
   • Auto use cache: False
   • API delay: 0.5s
   • Random seed: 42


## 4. Load Test Dataset from popular_dishes_30.json

In [9]:
# Load test dishes from popular_dishes_30.json
with open('popular_dishes_30.json', 'r', encoding='utf-8') as f:
    dish_list = json.load(f)

print(f"✅ Loaded {len(dish_list)} dishes from popular_dishes_30.json")

# Map dish_id to full dish data from KB
test_dishes = []
dish_id_map = {dish['id']: dish for dish in dish_kb}

for item in dish_list:
    dish_id = item['dish_id']
    if dish_id in dish_id_map:
        test_dishes.append(dish_id_map[dish_id])
    else:
        print(f"⚠️  Dish ID not found in KB: {dish_id} ({item['name_vi']})")

print(f"\n✅ Successfully loaded {len(test_dishes)} dishes for testing")

# Show all dishes
print("\n📋 Danh sách 30 món ăn:")
print("=" * 80)
for i, dish in enumerate(test_dishes, 1):
    ing_count = len(dish.get('ingredients', []))
    category = dish.get('category', 'unknown')
    print(f"{i:2}. {dish.get('name_vi'):50} ({ing_count:2} NL, {category})")
print("=" * 80)

✅ Loaded 30 dishes from popular_dishes_30.json

✅ Successfully loaded 30 dishes for testing

📋 Danh sách 30 món ăn:
 1. Phở chiên phồng bò xào                             (16 NL, mon chien)
 2. Bún măng vịt  bằng nồi cơm điện tử                 (11 NL, mon nuoc)
 3. Phở xào tim gà                                     (13 NL, mon xao)
 4. Bún xào rau cải thịt heo                           (10 NL, mon xao)
 5. Bún gạo xào lòng gà                                (13 NL, mon xao)
 6. Bún gỏi dà Sóc Trăng                               (13 NL, mon nuoc)
 7. Bánh khoai mì nướng bằng nồi cơm điện         (12 NL, mon banh)
 8. Cơm chiên bò xào                                   (17 NL, mon chien)
 9. Cách làm đậu hũ xào thịt sốt cà chua đơn giản, bắt cơm (12 NL, mon xao)
10. Gỏi cá cơm tươi                                    (15 NL, mon goi - salad)
11. Nấu lẩu bằng nồi cơm điện                          (20 NL, mon lau)
12. Canh bún bạch tuộc                                 (17 NL, mon nuoc)


## 5. Run RAG Predictions and Save to Cache

⚠️ **CHI PHÍ**: 
- Test dataset: **30 món ăn**
- Mỗi món gọi 1 lần RAG (Bedrock KB + Nova model)
- Ước tính thời gian: ~15 phút (0.5s delay/món)
- Kết quả lưu vào `predictions_cache.json`

In [10]:
# Check if cached predictions exist
USE_CACHE = USE_CACHE_AUTO and Path(CACHE_FILE).exists()

if USE_CACHE:
    print(f"🔍 Found cached predictions in '{CACHE_FILE}'")
    if not USE_CACHE_AUTO:
        response = input("Load from cache? (y/n): ").strip().lower()
        USE_CACHE = response == 'y'
    else:
        print(f"📂 AUTO-LOADING from cache (set USE_CACHE_AUTO=False to disable)")

if USE_CACHE:
    print(f"📥 Loading cached predictions...")
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        cached_data = json.load(f)
        predictions = cached_data['predictions']
        errors = cached_data.get('errors', [])
    print(f"✅ Loaded {len(predictions)} predictions from cache")
    print(f"   Cached at: {cached_data.get('timestamp', 'unknown')}")
else:
    # Run predictions
    print(f"\n{'='*80}")
    print(f"🚀 STARTING RAG PREDICTIONS")
    print(f"{'='*80}")
    print(f"📊 Processing {len(test_dishes)} dishes...")
    print(f"⏱️  Estimated time: ~{len(test_dishes) * API_DELAY / 60:.1f} minutes")
    print(f"💰 API Calls: ~{len(test_dishes)} RAG queries")
    print(f"{'='*80}\n")
    
    predictions = []
    errors = []

    for idx, dish in enumerate(tqdm(test_dishes, desc="🔄 Processing dishes"), 1):
        dish_name = dish.get('name_vi')
        
        try:
            # Call KB service to get RAG prediction
            rag_result = kb_service.get_dish_recipe(dish_name)
            
            # Normalize ingredient IDs using ingredient resolver
            normalized_ingredients = []
            for ing in rag_result.get('ingredients', []):
                vietnamese_name = ing.get('vietnamese_name', '')
                
                # Resolve to ingredient ID
                ing_id = ingredient_resolver.resolve_name_to_id(vietnamese_name)
                
                if ing_id:
                    ing_info = ontology.get_ingredient(ing_id) or {}
                    normalized_ingredients.append({
                        'ingredient_id': ing_id,
                        'name_vi': vietnamese_name,
                        'quantity': ing.get('quantity', ''),
                        'unit': ing.get('unit', ''),
                        'category': ing_info.get('category', '')
                    })
            
            predictions.append({
                'dish_name': dish_name,
                'ground_truth': dish,
                'rag_prediction': {
                    'vietnamese_name': rag_result.get('vietnamese_name', dish_name),
                    'name': rag_result.get('name', ''),
                    'ingredients': normalized_ingredients
                }
            })
            
            # Delay to avoid rate limiting
            time.sleep(API_DELAY)
            
        except Exception as e:
            errors.append({
                'dish_name': dish_name,
                'error': str(e)
            })
            print(f"\n❌ Error processing '{dish_name}': {e}")

    print(f"\n{'='*80}")
    print(f"✅ PREDICTIONS COMPLETED")
    print(f"{'='*80}")
    print(f"   ✓ Successful: {len(predictions)}")
    print(f"   ✗ Errors: {len(errors)}")
    print(f"{'='*80}\n")
    
    # Save to cache
    print(f"💾 Saving predictions to cache...")
    cache_data = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'sample_size': len(test_dishes),
        'config': {
            'SAMPLE_SIZE': SAMPLE_SIZE,
            'MIN_IMPORTANCE_CORE': MIN_IMPORTANCE_CORE,
            'RANDOM_SEED': RANDOM_SEED,
            'source_file': 'popular_dishes_30.json'
        },
        'predictions': predictions,
        'errors': errors
    }
    
    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(cache_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Cached to '{CACHE_FILE}' for future use")

# Final summary
print(f"\n{'='*80}")
print(f"🎉 RAG EXTRACTION COMPLETED!")
print(f"{'='*80}")
print(f"📊 Summary:")
print(f"   • Total dishes processed:  {len(test_dishes)}")
print(f"   • Successful predictions:  {len(predictions)}")
print(f"   • Failed predictions:      {len(errors)}")
print(f"   • Cache file:              {CACHE_FILE}")
print(f"   • Source file:             popular_dishes_30.json")
print(f"{'='*80}")

if errors:
    print(f"\n⚠️  Errors encountered:")
    for err in errors:
        print(f"   • {err['dish_name']}: {err['error']}")


🚀 STARTING RAG PREDICTIONS
📊 Processing 30 dishes...
⏱️  Estimated time: ~0.2 minutes
💰 API Calls: ~30 RAG queries



🔄 Processing dishes: 100%|██████████| 30/30 [08:13<00:00, 16.45s/it]


✅ PREDICTIONS COMPLETED
   ✓ Successful: 30
   ✗ Errors: 0

💾 Saving predictions to cache...
✅ Cached to 'predictions_cache_v1.json' for future use

🎉 RAG EXTRACTION COMPLETED!
📊 Summary:
   • Total dishes processed:  30
   • Successful predictions:  30
   • Failed predictions:      0
   • Cache file:              predictions_cache_v1.json
   • Source file:             popular_dishes_30.json
